# Test Search Qdrant Tourism

Notebook này dùng để test các kiểu tìm kiếm trong project:

1. **Tìm ảnh bằng ảnh**: ảnh input → SigLIP image vector → search `image_vector`
2. **Tìm ảnh bằng text kiểu multimodal**: text input → SigLIP text vector → search `image_vector`
3. **Tìm ảnh bằng text theo caption semantic**: text input → BGE-M3 → search `caption_vector`
4. **Hybrid search ảnh bằng text**: kết hợp điểm từ SigLIP multimodal và BGE caption
5. **Tìm đoạn text/chunk liên quan**: text input → BGE-M3 → search `text_vector`

Nên đặt file `.ipynb` này ở **thư mục gốc project** `tourism-embedding-qdrant/`, cùng cấp với `main.py`.


In [1]:
# =========================
# 1. Import và load project
# =========================

import os
import sys
from pathlib import Path
from typing import Dict, Any, List, Optional
from io import BytesIO

from dotenv import load_dotenv
import yaml
from IPython.display import display, Image as IPyImage
from PIL import Image

# Nếu notebook nằm trong thư mục gốc project thì PROJECT_ROOT là thư mục hiện tại.
PROJECT_ROOT = Path.cwd()

# Nếu bạn đặt notebook ở chỗ khác, sửa lại dòng này:
# PROJECT_ROOT = Path(r"C:/Users/ASUS/Downloads/tourism-embedding-qdrant")

sys.path.append(str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

from src.qdrant_manager import QdrantManager
from src.embedding_models import EmbeddingModels
from src.s3_loader import get_s3_client, read_image_from_s3

print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: c:\Users\ASUS\Downloads\tourism-embedding-qdrant


In [2]:
# =========================
# 2. Load config
# =========================

def resolve_env_value(value):
    """Đổi ${ENV_NAME} thành giá trị trong .env."""
    if isinstance(value, str) and value.startswith("${") and value.endswith("}"):
        env_name = value[2:-1]
        return os.getenv(env_name)
    return value


def load_config(config_path: str = "configs/config.yaml"):
    config_file = PROJECT_ROOT / config_path

    with open(config_file, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)

    config["s3"]["bucket"] = resolve_env_value(config["s3"]["bucket"])
    config["qdrant"]["url"] = resolve_env_value(config["qdrant"]["url"])

    return config


config = load_config()

print("S3 bucket:", config["s3"]["bucket"])
print("Qdrant URL:", config["qdrant"]["url"])
print("Image collection:", config["collections"]["image_collection"])
print("Text collection:", config["collections"]["text_collection"])


S3 bucket: gia-lai-tourism-images
Qdrant URL: http://localhost:6333
Image collection: image_collection
Text collection: text_collection


In [3]:
# =========================
# 3. Khởi tạo Qdrant, S3, Models
# =========================

qdrant = QdrantManager(
    url=config["qdrant"]["url"],
    location_collection=config["collections"]["location_info"],
    image_collection=config["collections"]["image_collection"],
    text_collection=config["collections"]["text_collection"],
)

s3_client = get_s3_client()
bucket = config["s3"]["bucket"]

models = EmbeddingModels(
    siglip_model_name=config["models"]["siglip"],
    florence_model_name=config["models"]["florence"],
    bge_model_name=config["models"]["bge"],
    translator_model_name=config["models"].get("translator"),
    use_translator=bool(config["models"].get("use_translator", True)),
)


Device: cpu
Đang load SigLIP...
Đang load Florence-2...
Đang load BGE-M3...
Đang load model dịch EN -> VI...
Đang load model sparse embedding...
Load model xong.


In [4]:
# =========================
# 4. Helper hiển thị ảnh từ S3
# =========================

def parse_s3_path(s3_path: str):
    """
    Hỗ trợ các dạng:
    1) vietnam-tourism/HON_KHO/image/00008.jpg
    2) s3://vietnam-tourism/HON_KHO/image/00008.jpg

    Trả về:
    bucket, s3_key
    """

    if not s3_path:
        return None, None

    s3_path = s3_path.strip()

    if s3_path.startswith("s3://"):
        s3_path = s3_path.replace("s3://", "", 1)

    parts = s3_path.split("/", 1)

    if len(parts) != 2:
        return None, None

    bucket_name = parts[0]
    s3_key = parts[1]

    return bucket_name, s3_key


def generate_presigned_image_url(
    s3_client,
    s3_path: str,
    expires_in: int = 3600
):
    """
    Tạo URL https tạm thời từ s3_path.

    Ví dụ s3_path:
    vietnam-tourism/HON_KHO/image/00008_2a069b63edbebb92.jpg
    """

    bucket_name, s3_key = parse_s3_path(s3_path)

    if not bucket_name or not s3_key:
        raise ValueError(f"s3_path không hợp lệ: {s3_path}")

    return s3_client.generate_presigned_url(
        ClientMethod="get_object",
        Params={
            "Bucket": bucket_name,
            "Key": s3_key,
        },
        ExpiresIn=expires_in,
    )


def show_s3_image(payload: Dict[str, Any], width: int = 320):
    """
    Hiển thị ảnh từ payload Qdrant.
    Hiện tại payload chỉ cần có field:
    s3_path = vietnam-tourism/HON_KHO/image/xxx.jpg
    """

    s3_path = payload.get("s3_path") or payload.get("path")

    if not s3_path:
        print("Không có s3_path trong payload")
        return

    url = generate_presigned_image_url(
        s3_client=s3_client,
        s3_path=s3_path,
    )

    display(IPyImage(url=url, width=width))


def print_image_result(point, rank: int, show_image: bool = True):
    payload = point.payload or {}
    score = float(point.score)

    s3_path = payload.get("s3_path") or payload.get("path")

    print(f"Top {rank} | Score: {score:.4f}")
    print(f"Image ID      : {payload.get('image_id')}")
    print(f"File          : {payload.get('file_name')}")
    print(f"Location      : {payload.get('location_name')} | {payload.get('province')}")
    print(f"Caption VI    : {payload.get('caption_vi')}")
    print(f"S3 path       : {s3_path}")
    print("-" * 80)

    if show_image:
        show_s3_image(payload)


def print_text_result(point, rank: int):
    payload = point.payload or {}
    score = float(point.score)

    print(f"Top {rank} | Score: {score:.4f}")
    print(f"Location      : {payload.get('location_name')} | {payload.get('province')}")
    print(f"File          : {payload.get('source_file')}")
    print(f"Document type : {payload.get('document_type')}")
    print(f"Section       : {payload.get('section_number')} - {payload.get('section_title')}")
    print(f"Chunk         : {payload.get('chunk_index')} / {payload.get('total_chunks')}")
    print("Content:")
    print((payload.get("content") or "")[:1200])
    print("-" * 80)

## 5. Tìm ảnh bằng ảnh input

Dùng khi người dùng upload một ảnh.

```txt
ảnh input → SigLIP image embedding → search image_vector trong image_collection
```


In [ ]:
# =========================
# Search ảnh bằng ảnh local
# =========================

def search_images_by_local_image(image_path: str, top_k: int = 5, show_image: bool = True):
    image = Image.open(image_path).convert("RGB")
    query_vector = models.encode_images_batch(image)

    results = qdrant.search_images_by_image_vector(
        query_vector=query_vector,
        top_k=top_k,
    )

    print(f"Search ảnh bằng ảnh input: {image_path}")
    print("=" * 80)

    for i, point in enumerate(results, start=1):
        print_image_result(point, i, show_image=show_image)

    return results




In [ ]:
results = search_images_by_local_image(
    image_path=r"C:/Users/ASUS/Downloads/honkho.webp",
    top_k=3,
)


Search ảnh bằng ảnh input: C:/Users/ASUS/Downloads/honkho.webp
Top 1 | Score: 0.8675
Image ID      : LOC_012_00117_c4ff452f3ea264c6
File          : 00117_c4ff452f3ea264c6.jpg
Location      : Hòn Khô | None
Caption VI    : Có mấy chiếc thuyền trong nước, có một ngọn đồi gần mé nước, có những hòn đá trên đó, có người đứng trên hòn đá trước hòn đá.
S3 path       : vietnam-tourism/HON_KHO/image/00117_c4ff452f3ea264c6.jpg
--------------------------------------------------------------------------------


Top 2 | Score: 0.8251
Image ID      : LOC_012_00050_dc4e6733abd69011
File          : 00050_dc4e6733abd69011.jpg
Location      : Hòn Khô | None
Caption VI    : Có nhiều thuyền trong nước , Có hòn đá lớn nơi bờ biển ; Có cây_cối ở trước các thuyền .
S3 path       : vietnam-tourism/HON_KHO/image/00050_dc4e6733abd69011.jpg
--------------------------------------------------------------------------------


Top 3 | Score: 0.8168
Image ID      : LOC_012_00049_c905e7cad1742729
File          : 00049_c905e7cad1742729.jpg
Location      : Hòn Khô | None
Caption VI    : Một nhóm người trên bờ biển, có hai chiếc thuyền dưới nước, có một ngọn đồi gần bờ biển với những tảng đá trên đó.
S3 path       : vietnam-tourism/HON_KHO/image/00049_c905e7cad1742729.jpg
--------------------------------------------------------------------------------


## 6. Tìm ảnh bằng text kiểu multimodal

Dùng SigLIP text embedding để search trực tiếp trên `image_vector`.

```txt
text input → SigLIP text embedding → search image_vector
```

Kiểu này hợp với mô tả thị giác ngắn, ví dụ: `bãi biển xanh`, `cầu gỗ ven biển`, `nước biển trong xanh`.


In [36]:
# =========================
# Text -> SigLIP text vector -> image_vector
# =========================

def search_images_by_text_multimodal(query_text: str, top_k: int = 5, show_image: bool = True):
    query_vector = models.encode_text_siglip(query_text)

    results = qdrant.search_images_by_image_vector(
        query_vector=query_vector,
        top_k=top_k,
    )

    print(f"Search ảnh bằng text multimodal/SigLIP: {query_text}")
    print("=" * 80)

    for i, point in enumerate(results, start=1):
        print_image_result(point, i, show_image=show_image)

    return results





In [28]:
from pathlib import Path
from datetime import datetime


def save_vector_to_txt(
    query_text: str,
    query_vector,
    output_dir: str = "vector_outputs",
    file_prefix: str = "siglip_text_vector"
):
    """
    Lưu full vector đầu vào ra file txt.
    Không đánh số index.
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = output_dir / f"{file_prefix}_{timestamp}.txt"

    # Nếu query_vector là numpy array thì chuyển sang list
    if hasattr(query_vector, "tolist"):
        vector_list = query_vector.tolist()
    else:
        vector_list = list(query_vector)

    with open(output_path, "w", encoding="utf-8") as f:
        f.write("QUERY TEXT\n")
        f.write("=" * 80 + "\n")
        f.write(query_text + "\n\n")

        f.write("VECTOR INFO\n")
        f.write("=" * 80 + "\n")
        f.write(f"Type: {type(query_vector)}\n")
        f.write(f"Dimension: {len(vector_list)}\n\n")

        f.write("FULL VECTOR\n")
        f.write("=" * 80 + "\n")
        f.write(str(vector_list))

    return output_path

In [29]:
# =========================
# Text -> SigLIP text vector -> image_vector
# =========================

def search_images_by_text_multimodal(
    query_text: str,
    top_k: int = 5,
    show_image: bool = True,
    show_vector: bool = True,
    vector_preview: int = 20,
    save_vector: bool = True,
    output_dir: str = "vector_outputs"
):
    query_vector = models.encode_text_siglip(query_text)

    if show_vector:
        print("INPUT TEXT")
        print("=" * 80)
        print(query_text)
        print()

        print("SIGLIP TEXT QUERY VECTOR")
        print("=" * 80)
        print(f"Type      : {type(query_vector)}")
        print(f"Dimension : {len(query_vector)}")
        print(f"Preview {vector_preview} values:")
        print(query_vector[:vector_preview])
        print("=" * 80)

    if save_vector:
        output_path = save_vector_to_txt(
            query_text=query_text,
            query_vector=query_vector,
            output_dir=output_dir,
            file_prefix="siglip_text_vector"
        )

        print(f"Đã lưu vector vào file: {output_path}")
        print("=" * 80)

    results = qdrant.search_images_by_image_vector(
        query_vector=query_vector,
        top_k=top_k,
    )

    print(f"Search ảnh bằng text multimodal/SigLIP: {query_text}")
    print("=" * 80)

    for i, point in enumerate(results, start=1):
        print_image_result(point, i, show_image=show_image)

    return results

In [30]:
results = search_images_by_text_multimodal(
    query_text="hòn khô được biết đến với những điểm nổi bật nào?",
    top_k=5,
    show_image=True,
    show_vector=True,
    vector_preview=20,
    save_vector=True,
    output_dir="vector_outputs"
)

INPUT TEXT
hòn khô được biết đến với những điểm nổi bật nào?

SIGLIP TEXT QUERY VECTOR
Type      : <class 'list'>
Dimension : 768
Preview 20 values:
[-0.023181229829788208, -0.0005355096654966474, 0.007058342453092337, -0.009722172282636166, 0.01671249233186245, -0.010311969555914402, -0.010836560279130936, 0.019819587469100952, 0.0030811158940196037, -0.010159107856452465, 0.006073580589145422, 0.0026586197782307863, -0.0069484952837228775, 0.02233865112066269, 0.0078447125852108, 0.02997954562306404, -0.00710098585113883, -0.006058624479919672, -0.009973864071071148, -0.010746183805167675]
Đã lưu vector vào file: vector_outputs\siglip_text_vector_20260531_154136.txt
Search ảnh bằng text multimodal/SigLIP: hòn khô được biết đến với những điểm nổi bật nào?
Top 1 | Score: 0.1264
Image ID      : LOC_012_00009_621e67d06a281c89
File          : 00009_621e67d06a281c89.jpg
Location      : Hòn Khô | None
Caption VI    : Đây là một hình_tượng của bờ_cõi : nước có màu xanh và rõ_ràng : có những 

Top 2 | Score: 0.1264
Image ID      : LOC_011_Bản sao của 00063_820946bb3a4a5a95
File          : Bản sao của 00063_820946bb3a4a5a95.jpg
Location      : Biển Quy Hòa | None
Caption VI    : Ấy_là một hình_tượng của bãi_biển , có nhiều người ở đó ; nước thì xanh và yên_lặng ; bầu_trời màu trắng , bầu_trời thì xanh và mây trắng ; có những núi lớn ở trong nền , các núi đầy cây_cối xanh , cát trên bờ biển có ánh_sáng . Có một tòa nhà trắng ở bên bờ bờ biển .
S3 path       : vietnam-tourism/BIEN_QUY_HOA/image/Bản sao của 00063_820946bb3a4a5a95.jpg
--------------------------------------------------------------------------------


Top 3 | Score: 0.1254
Image ID      : LOC_012_00039_d35a78f87dc962cf
File          : 00039_d35a78f87dc962cf.jpg
Location      : Hòn Khô | None
Caption VI    : Hình_tượng về bờ_cõi biển , nước là xanh và trong_sạch ; có những tàu nhỏ ở trong nước ; bên hữu mặt bàn_thờ có một cái bãi có nhà ở trên đó . Các nhà đều có những ngôi_sao ; bầu_trời đầy những mây trắng ; những núi ở trong có những cây xanh_tươi bao_phủ , cát nơi bờ biển đều trắng .
S3 path       : vietnam-tourism/HON_KHO/image/00039_d35a78f87dc962cf.jpg
--------------------------------------------------------------------------------


Top 4 | Score: 0.1243
Image ID      : LOC_012_00112_32592988776b686a
File          : 00112_32592988776b686a.jpg
Location      : Hòn Khô | None
Caption VI    : Hình_tượng ấy_là một khoảng từ một hòn đảo ở giữa biển ; nước màu xanh và xanh . Có những sóng trắng nhỏ trong nước ; hòn đảo màu xanh và cỏ xanh ; có một bãi cát trên hòn đảo ; có ánh sáng và một vài ngôi nhà nhỏ trên bờ biển .
S3 path       : vietnam-tourism/HON_KHO/image/00112_32592988776b686a.jpg
--------------------------------------------------------------------------------


Top 5 | Score: 0.1204
Image ID      : LOC_012_00108_809ebaa7f8cb2555
File          : 00108_809ebaa7f8cb2555.jpg
Location      : Hòn Khô | None
Caption VI    : Hòn đảo ấy xem như một hòn đảo, nước bao phủ bởi nước; nước màu xanh và yên lặng; có một mống trên bầu trời cao trên hòn đảo; trên núi có mây trắng; trên núi có những ngôi nhà, ở đó có tường bao quanh những phòng, có tường bao bọc bầu trời màu trắng.
S3 path       : vietnam-tourism/HON_KHO/image/00108_809ebaa7f8cb2555.jpg
--------------------------------------------------------------------------------


In [37]:
# Ví dụ:
results = search_images_by_text_multimodal(
    query_text="hòn khô",
    top_k=3,
)

Search ảnh bằng text multimodal/SigLIP: hòn khô
Top 1 | Score: 0.0383
Image ID      : LOC_012_00060_5dea505f44dd3df2
File          : 00060_5dea505f44dd3df2.jpg
Location      : Hòn Khô | None
Caption VI    : Một người đờn bà mặc áo xanh bước đi trên một đường bằng gỗ; người đờn bà đội một cái nón màu nâu; có hòn đá lớn trên lối đi bên cạnh người đờn bà; có một khối nước đằng sau người đờn bà và người đờn bà ở trên đó, trên bầu trời có mây màu xanh.
S3 path       : vietnam-tourism/HON_KHO/image/00060_5dea505f44dd3df2.jpg
--------------------------------------------------------------------------------


Top 2 | Score: 0.0339
Image ID      : LOC_012_00041_b375ed18cc19b78d
File          : 00041_b375ed18cc19b78d.jpg
Location      : Hòn Khô | None
Caption VI    : Người đờn bà đứng trên cây thông bằng gỗ, mặc một cái áo dài bằng cam và một cái mũ rơm. Người đờn bà cầm trong tay một cái túi trắng, và cái cầu thì đầy đá lớn; nước thì xanh và yên lặng; có một ngọn núi đằng sau người đờn bà, và có một bầu trời màu trắng.
S3 path       : vietnam-tourism/HON_KHO/image/00041_b375ed18cc19b78d.jpg
--------------------------------------------------------------------------------


Top 3 | Score: 0.0217
Image ID      : LOC_012_00004_19ebd98aaa7c4f29
File          : 00004_19ebd98aaa7c4f29.jpg
Location      : Hòn Khô | None
Caption VI    : Có hai người đứng trên một cái tàu bằng gỗ: người nữ mặc áo sơ mi đỏ và đội mũ trắng; một người đờn bà khác mặc áo dài màu vàng, có đáy đen, bao quanh bởi những hòn đá lớn; nước thì xanh và yên lặng; có một ngọn đồi nhỏ ở bên kia nước, và bầu trời thì có mây trắng.
S3 path       : vietnam-tourism/HON_KHO/image/00004_19ebd98aaa7c4f29.jpg
--------------------------------------------------------------------------------


## 7. Tìm ảnh bằng text theo caption semantic

Dùng BGE-M3 để embedding query, rồi search trên `caption_vector`.

```txt
text input → BGE-M3 → search caption_vector
```

Kiểu này hợp với truy vấn ngữ nghĩa hơn, ví dụ: `địa điểm phù hợp đi biển ở Quy Nhơn`, `ảnh về Hòn Khô`, `khu du lịch có san hô`.


In [38]:
# =========================
# Text -> BGE-M3 -> caption_vector
# =========================

def search_images_by_caption_text(query_text: str, top_k: int = 5, show_image: bool = True):
    query_vector = models.encode_text_bge(query_text)

    results = qdrant.search_images_by_caption_vector(
        query_vector=query_vector,
        top_k=top_k,
    )

    print(f"Search ảnh bằng caption semantic/BGE: {query_text}")
    print("=" * 80)

    for i, point in enumerate(results, start=1):
        print_image_result(point, i, show_image=show_image)

    return results




In [39]:

# Ví dụ:
results = search_images_by_caption_text(
    query_text="cho tôi ảnh liên quan đến Bình Định",
    top_k=3,
)

Search ảnh bằng caption semantic/BGE: cho tôi ảnh liên quan đến Bình Định
Top 1 | Score: 0.5858
Image ID      : LOC_011_Bản sao của 00023_1ffb1ee178762b63
File          : Bản sao của 00023_1ffb1ee178762b63.jpg
Location      : Biển Quy Hòa | None
Caption VI    : Ấy_là một hình_tượng của bờ_cõi nầy : nước có màu xanh và trắng ; các lượn sóng vỡ trên bờ biển ; có những sóng trắng nhỏ trong nước ; cát là màu đen , có một gò ở bên kia nước , trên đó có cây_cối xanh và xanh_tươi ; trên những bầu_trời có màu xanh_tươi .
S3 path       : vietnam-tourism/BIEN_QUY_HOA/image/Bản sao của 00023_1ffb1ee178762b63.jpg
--------------------------------------------------------------------------------


Top 2 | Score: 0.5825
Image ID      : LOC_011_Bản sao của 00002_8b25b98b6067dfc5
File          : Bản sao của 00002_8b25b98b6067dfc5.jpg
Location      : Biển Quy Hòa | None
Caption VI    : Ấy_là một hình_tượng của bãi , bầu_trời trong và xanh , có những mây trắng nhỏ trong bầu_trời ; mây trắng và đen ; nước là màu xanh , các lượn sóng thì đóng trên cát ; cát là hơi sáng , có một hòn đảo nhỏ trong khoảng_không ; các chê - ru-bin ấy đều có cây_cối ở trên nó .
S3 path       : vietnam-tourism/BIEN_QUY_HOA/image/Bản sao của 00002_8b25b98b6067dfc5.jpg
--------------------------------------------------------------------------------


Top 3 | Score: 0.5816
Image ID      : LOC_011_Bản sao của 00042_4258d830d98139b9
File          : Bản sao của 00042_4258d830d98139b9.jpg
Location      : Biển Quy Hòa | None
Caption VI    : Ấy_là một hình_tượng của bãi_biển , bờ_biển làm_nên bằng cát ; cát bằng vàng , và có những sóng nhỏ trong nước ; nước màu xanh , các lượn sóng trắng ; bầu_trời là bóng ; mây thì trắng , có cây_cối thì xanh và; có một cái chòi ở bên hữu bờ biển khỏe_mạnh .
S3 path       : vietnam-tourism/BIEN_QUY_HOA/image/Bản sao của 00042_4258d830d98139b9.jpg
--------------------------------------------------------------------------------


## 8. Hybrid search ảnh bằng text

Kết hợp 2 nhánh:

```txt
Nhánh 1: query text → SigLIP text vector → search image_vector
Nhánh 2: query text → BGE-M3 vector       → search caption_vector
```

Công thức điểm:

```txt
final_score = alpha * multimodal_score + (1 - alpha) * caption_score
```

Gợi ý:

```txt
alpha = 0.5  cân bằng
alpha = 0.6  ưu tiên ảnh/thị giác
alpha = 0.4  ưu tiên caption/ngữ nghĩa
```


In [41]:
# =========================
# Hybrid search: SigLIP image_vector + BGE caption_vector
# =========================

def hybrid_search_images_by_text(
    query_text: str,
    top_k: int = 5,
    candidate_k: int = 20,
    alpha: float = 0.3,
    show_image: bool = True,
):
    siglip_query_vector = models.encode_text_siglip(query_text)
    bge_query_vector = models.encode_text_bge(query_text)

    siglip_results = qdrant.search_images_by_image_vector(
        query_vector=siglip_query_vector,
        top_k=candidate_k,
    )

    caption_results = qdrant.search_images_by_caption_vector(
        query_vector=bge_query_vector,
        top_k=candidate_k,
    )

    merged: Dict[str, Dict[str, Any]] = {}

    def get_merge_key(point):
        payload = point.payload or {}
        return payload.get("image_id") or payload.get("s3_key") or str(point.id)

    def add_result(point, source: str):
        key = get_merge_key(point)
        payload = point.payload or {}

        if key not in merged:
            merged[key] = {
                "point": point,
                "payload": payload,
                "multimodal_score": 0.0,
                "caption_score": 0.0,
                "sources": [],
            }

        if source == "multimodal":
            merged[key]["multimodal_score"] = max(
                merged[key]["multimodal_score"],
                float(point.score),
            )
        elif source == "caption":
            merged[key]["caption_score"] = max(
                merged[key]["caption_score"],
                float(point.score),
            )

        merged[key]["sources"].append(source)

    for point in siglip_results:
        add_result(point, "multimodal")

    for point in caption_results:
        add_result(point, "caption")

    final_results = []

    for key, item in merged.items():
        multimodal_score = item["multimodal_score"]
        caption_score = item["caption_score"]

        final_score = alpha * multimodal_score + (1 - alpha) * caption_score

        final_results.append({
            "key": key,
            "point": item["point"],
            "payload": item["payload"],
            "multimodal_score": multimodal_score,
            "caption_score": caption_score,
            "final_score": final_score,
            "sources": sorted(set(item["sources"])),
        })

    final_results = sorted(
        final_results,
        key=lambda x: x["final_score"],
        reverse=True,
    )[:top_k]

    print(f"Hybrid search ảnh bằng text: {query_text}")
    print(f"Formula: final_score = {alpha} * multimodal_score + {1-alpha:.2f} * caption_score")
    print("=" * 80)

    for i, item in enumerate(final_results, start=1):
        payload = item["payload"]

        print(f"Top {i} | Final: {item['final_score']:.4f} | "
              f"Multimodal: {item['multimodal_score']:.4f} | "
              f"Caption: {item['caption_score']:.4f} | "
              f"Sources: {item['sources']}")
        print(f"Image ID   : {payload.get('image_id')}")
        print(f"File       : {payload.get('file_name')}")
        print(f"Location   : {payload.get('location_name')} | {payload.get('province')}")
        print(f"Caption VI : {payload.get('caption_vi')}")
        print(f"S3 path    : {payload.get('s3_path') or payload.get('path')}")
        print("-" * 80)

        if show_image:
            show_s3_image(payload)

    return final_results




In [42]:
# Ví dụ:
results = hybrid_search_images_by_text(
query_text="Cho tôi ảnh liên quan đến đảo ở Quy Nhơn",
    top_k=5,
    candidate_k=20,
    alpha=0.3,
)


Hybrid search ảnh bằng text: Cho tôi ảnh liên quan đến đảo ở Quy Nhơn
Formula: final_score = 0.3 * multimodal_score + 0.70 * caption_score
Top 1 | Final: 0.4545 | Multimodal: 0.0000 | Caption: 0.6493 | Sources: ['caption']
Image ID   : LOC_012_00112_32592988776b686a
File       : 00112_32592988776b686a.jpg
Location   : Hòn Khô | None
Caption VI : Hình_tượng ấy_là một khoảng từ một hòn đảo ở giữa biển ; nước màu xanh và xanh . Có những sóng trắng nhỏ trong nước ; hòn đảo màu xanh và cỏ xanh ; có một bãi cát trên hòn đảo ; có ánh sáng và một vài ngôi nhà nhỏ trên bờ biển .
S3 path    : vietnam-tourism/HON_KHO/image/00112_32592988776b686a.jpg
--------------------------------------------------------------------------------


Top 2 | Final: 0.4536 | Multimodal: 0.0185 | Caption: 0.6400 | Sources: ['caption', 'multimodal']
Image ID   : LOC_012_00004_19ebd98aaa7c4f29
File       : 00004_19ebd98aaa7c4f29.jpg
Location   : Hòn Khô | None
Caption VI : Có hai người đứng trên một cái tàu bằng gỗ: người nữ mặc áo sơ mi đỏ và đội mũ trắng; một người đờn bà khác mặc áo dài màu vàng, có đáy đen, bao quanh bởi những hòn đá lớn; nước thì xanh và yên lặng; có một ngọn đồi nhỏ ở bên kia nước, và bầu trời thì có mây trắng.
S3 path    : vietnam-tourism/HON_KHO/image/00004_19ebd98aaa7c4f29.jpg
--------------------------------------------------------------------------------


Top 3 | Final: 0.4512 | Multimodal: 0.0000 | Caption: 0.6446 | Sources: ['caption']
Image ID   : LOC_012_00043_0c440b0bd9c9d6db
File       : 00043_0c440b0bd9c9d6db.jpg
Location   : Hòn Khô | None
Caption VI : Ấy đó là một hình_tượng của biển , nước thì xanh và trong_sạch . Có những sóng nhỏ trong nước , những lượn sóng trắng , cát trên bãi_biển là màu đen ; có một hòn đá trên mặt_đất , và những vầng đá ấy là ánh_sáng ; những hòn đá thì ở trong những đám mây trắng , và mặt_trời thì che_phủ cát .
S3 path    : vietnam-tourism/HON_KHO/image/00043_0c440b0bd9c9d6db.jpg
--------------------------------------------------------------------------------


Top 4 | Final: 0.4491 | Multimodal: 0.0000 | Caption: 0.6416 | Sources: ['caption', 'multimodal']
Image ID   : LOC_012_00110_78b5464b003a470c
File       : 00110_78b5464b003a470c.jpg
Location   : Hòn Khô | None
Caption VI : Ấy_là một hình_tượng của biển ấy : nước là xanh và xanh ; có những sóng nhỏ trong nước ; các lượn đều trắng , người_ta đi trên cát , cát là ánh_sáng ; có một cái thuyền nhỏ trắng và xanh ; bầu_trời có mây trắng , các gò xanh và đen .
S3 path    : vietnam-tourism/HON_KHO/image/00110_78b5464b003a470c.jpg
--------------------------------------------------------------------------------


Top 5 | Final: 0.4476 | Multimodal: 0.0000 | Caption: 0.6395 | Sources: ['caption']
Image ID   : LOC_012_00009_621e67d06a281c89
File       : 00009_621e67d06a281c89.jpg
Location   : Hòn Khô | None
Caption VI : Đây là một hình_tượng của bờ_cõi : nước có màu xanh và rõ_ràng : có những tàu nhỏ ở trong nước , những thuyền trắng và xanh ; cát trên bờ biển có màu đen ; có một ngọn đồi đằng sau cát ; trên đó có những cây_cối xanh_tươi , trên đó có những từng mây trắng , và có hình_tượng nó bằng mây trắng .
S3 path    : vietnam-tourism/HON_KHO/image/00009_621e67d06a281c89.jpg
--------------------------------------------------------------------------------


## 9. Tìm đoạn text/chunk liên quan

Dùng BGE-M3 để search trên `text_vector`.

```txt
text query → BGE-M3 → search text_collection.text_vector
```


In [ ]:
# =========================
# Text -> BGE-M3 -> text_vector
# =========================

def search_text_chunks_by_query(query_text: str, top_k: int = 5):
    query_vector = models.encode_text_bge(query_text)

    results = qdrant.search_text_chunks_dense(
        query_vector=query_vector,
        top_k=top_k,
    )

    print(f"Search text chunks: {query_text}")
    print("=" * 80)

    for i, point in enumerate(results, start=1):
        print_text_result(point, i)

    return results


def search_text_chunks_sparse_notebook(query_text: str, top_k: int = 5):
    query_sparse_vector = models.encode_text_sparse(query_text)

    results = qdrant.search_text_chunks_sparse(
        query_sparse_vector=query_sparse_vector,
        top_k=top_k,
    )

    print(f"Search text chunks: {query_text}")
    print("=" * 80)

    for i, point in enumerate(results, start=1):
        print_text_result(point, i)

    return results1


In [5]:
def search_text_chunks_dense_notebook(query_text: str, top_k: int = 5):
    query_vector = models.encode_text_bge(query_text)

    results = qdrant.search_text_chunks_dense(
        query_vector=query_vector,
        top_k=top_k,
    )

    print(f"DENSE SEARCH TEXT: {query_text}")
    print("=" * 80)

    for i, point in enumerate(results, start=1):
        payload = point.payload or {}

        print(f"Top {i} | Score: {float(point.score):.4f}")
        print(f"Location : {payload.get('location_name')}")
        print(f"File     : {payload.get('source_file')}")
        print(f"Section  : {payload.get('section_number')} - {payload.get('section_title')}")
        print("Content:")
        print((payload.get("content") or "")[:1200])
        print("-" * 80)

    return results

In [ ]:
results = search_text_chunks_dense_notebook(
    query_text="hòn khô",
    top_k=5,

)

DENSE SEARCH TEXT: hòn khô
Top 1 | Score: 0.5850
Location : None
File     : surrounding.txt
Section  : 3 - Hoạt động theo mùa
Content:
Mùa khô từ tháng 4 đến tháng 9 phù hợp với tắm biển, cano, lặn ngắm san hô, chụp ảnh và tour trong ngày. Đây là mùa nên ưu tiên nếu người dùng hỏi lần đầu đi Hòn Khô nên chọn tháng nào.
Mùa rong mơ từ tháng 5 đến tháng 7 phù hợp với người thích chụp ảnh dưới nước, flycam, nội dung thiên nhiên và trải nghiệm khác biệt. Tuy nhiên cần nhắc du khách không khai thác, giẫm đạp hoặc làm hư hại thảm rong vì rong mơ có vai trò sinh thái.
Mùa mưa bão và gió mạnh không phù hợp với hoạt động biển. Nếu vẫn đến Quy Nhơn vào mùa này, nên có phương án thay thế như tham quan trong thành phố, ăn uống, bảo tàng, tháp Chăm hoặc các điểm ít phụ thuộc vào cano.
--------------------------------------------------------------------------------
Top 2 | Score: 0.5819
Location : None
File     : ticket_schedule.txt
Section  : 4 - Thời điểm nên tham quan
Content:
Thời điểm được nhiề

: 

In [22]:
def search_text_chunks_dense_notebook(
    query_text: str,
    top_k: int = 5,
    show_vector: bool = True,
    vector_preview: int = 20
):
    query_vector = models.encode_text_bge(query_text)

    if show_vector:
        print("INPUT TEXT:")
        print(query_text)
        print("=" * 80)

        print("DENSE QUERY VECTOR")
        print(f"Type       : {type(query_vector)}")
        print(f"Dimension  : {len(query_vector)}")
        print(f"Preview {vector_preview} values:")
        print(query_vector[:vector_preview])
        print("=" * 80)

    results = qdrant.search_text_chunks_dense(
        query_vector=query_vector,
        top_k=top_k,
    )

    print(f"DENSE SEARCH TEXT: {query_text}")
    print("=" * 80)

    for i, point in enumerate(results, start=1):
        payload = point.payload or {}

        print(f"Top {i} | Score: {float(point.score):.4f}")
        print(f"Location : {payload.get('location_name')}")
        print(f"File     : {payload.get('source_file')}")
        print(f"Section  : {payload.get('section_number')} - {payload.get('section_title')}")
        print("Content:")
        print((payload.get("content") or "")[:1200])
        print("-" * 80)

    return results

In [24]:
results = search_text_chunks_dense_notebook(
    query_text="hòn khô được biết đến với những điểm nổi bật nào?",
    top_k=5,
    # show_vector=True,
    # vector_preview=1024
)

INPUT TEXT:
hòn khô được biết đến với những điểm nổi bật nào?
DENSE QUERY VECTOR
Type       : <class 'list'>
Dimension  : 1024
Preview 20 values:
[0.05820721760392189, 0.005221210420131683, -0.05568621680140495, -0.011833293363451958, 0.0018765735439956188, -0.07213915139436722, -0.009163375943899155, 0.03875543922185898, -0.04347319155931473, -0.012384499423205853, 0.014399543404579163, 0.017692361027002335, 0.0026229703798890114, 0.020053120329976082, 0.0035916147753596306, 0.009719669818878174, 0.0351468101143837, 0.008258671499788761, 0.022646313533186913, 0.026380063965916634]
DENSE SEARCH TEXT: hòn khô được biết đến với những điểm nổi bật nào?
Top 1 | Score: 0.5603
Location : None
File     : overview.docx
Section  : 6 - Điểm nổi bật
Content:
Điểm nổi bật thứ nhất là rạn san hô gần bờ. Một số nguồn du lịch mô tả san hô tại Hòn Khô nằm ở vùng nước không quá sâu, phù hợp với hoạt động lặn ống thở khi biển êm và có hướng dẫn an toàn. Đây là lý do Hòn Khô thường được xếp vào nhóm điểm

In [13]:
def search_text_chunks_sparse_notebook(query_text: str, top_k: int = 5):
    query_sparse_vector = models.encode_text_sparse(query_text)

    results = qdrant.search_text_chunks_sparse(
        query_sparse_vector=query_sparse_vector,
        top_k=top_k,
    )

    print(f"SPARSE SEARCH TEXT: {query_text}")
    print("=" * 80)

    for i, point in enumerate(results, start=1):
        payload = point.payload or {}

        print(f"Top {i} | Score: {float(point.score):.4f}")
        print(f"Location : {payload.get('location_name')}")
        print(f"File     : {payload.get('source_file')}")
        print(f"Section  : {payload.get('section_number')} - {payload.get('section_title')}")
        print("Content:")
        print((payload.get("content") or "")[:1200])
        print("-" * 80)

    return results

In [16]:
sparse_results = search_text_chunks_sparse_notebook(
    query_text="hòn khô được biến đến với những điểm nổi bật nào?",
    top_k=3
)

SPARSE SEARCH TEXT: hòn khô được biến đến với những điểm nổi bật nào?
Top 1 | Score: 10.1139
Location : None
File     : overview.docx
Section  : 6 - Điểm nổi bật
Content:
Điểm nổi bật thứ nhất là rạn san hô gần bờ. Một số nguồn du lịch mô tả san hô tại Hòn Khô nằm ở vùng nước không quá sâu, phù hợp với hoạt động lặn ống thở khi biển êm và có hướng dẫn an toàn. Đây là lý do Hòn Khô thường được xếp vào nhóm điểm du lịch dành cho khách thích biển xanh, tắm biển và khám phá hệ sinh thái dưới nước.
Điểm nổi bật thứ hai là mùa rong mơ. Từ khoảng tháng 5 đến tháng 7, khu vực Hòn Khô và Nhơn Hải có những thảm rong mơ dưới nước, tạo hiệu ứng như cánh đồng vàng dưới biển. Đây là chất liệu hình ảnh rất phù hợp cho du lịch chụp ảnh, flycam, nội dung truyền thông và các câu trả lời về “mùa nào Hòn Khô đẹp nhất”.
Điểm nổi bật thứ ba là cầu gỗ hoặc đường ven đá nhìn ra biển. Đây là góc check-in quen thuộc của nhiều du khách, đặc biệt vào thời điểm nắng đẹp, nước xanh và trời quang. Tuy nhiên, tình tr

In [17]:
def hybrid_search_text_chunks_notebook(
    query_text: str,
    top_k: int = 3,
    candidate_k: int = 20,
    alpha: float = 0.5,
):
    dense_query_vector = models.encode_text_bge(query_text)
    sparse_query_vector = models.encode_text_sparse(query_text)

    dense_results = qdrant.search_text_chunks_dense(
        query_vector=dense_query_vector,
        top_k=candidate_k,
    )

    sparse_results = qdrant.search_text_chunks_sparse(
        query_sparse_vector=sparse_query_vector,
        top_k=candidate_k,
    )

    merged: Dict[str, Dict[str, Any]] = {}

    def get_key(point):
        payload = point.payload or {}
        return payload.get("chunk_id") or str(point.id)

    def add_result(point, source: str):
        key = get_key(point)
        payload = point.payload or {}

        if key not in merged:
            merged[key] = {
                "point": point,
                "payload": payload,
                "dense_score": 0.0,
                "sparse_score": 0.0,
                "sources": [],
            }

        if source == "dense":
            merged[key]["dense_score"] = max(
                merged[key]["dense_score"],
                float(point.score),
            )

        elif source == "sparse":
            merged[key]["sparse_score"] = max(
                merged[key]["sparse_score"],
                float(point.score),
            )

        merged[key]["sources"].append(source)

    for point in dense_results:
        add_result(point, "dense")

    for point in sparse_results:
        add_result(point, "sparse")

    final_results = []

    for key, item in merged.items():
        dense_score = item["dense_score"]
        sparse_score = item["sparse_score"]

        final_score = alpha * dense_score + (1 - alpha) * sparse_score

        final_results.append({
            "chunk_id": key,
            "point": item["point"],
            "payload": item["payload"],
            "dense_score": dense_score,
            "sparse_score": sparse_score,
            "final_score": final_score,
            "sources": sorted(set(item["sources"])),
        })

    final_results = sorted(
        final_results,
        key=lambda x: x["final_score"],
        reverse=True,
    )[:top_k]

    print(f"HYBRID TEXT SEARCH: {query_text}")
    print(f"Formula: final_score = {alpha} * dense_score + {1-alpha:.2f} * sparse_score")
    print("=" * 80)

    for i, item in enumerate(final_results, start=1):
        payload = item["payload"]

        print(
            f"Top {i} | "
            f"Final: {item['final_score']:.4f} | "
            f"Dense: {item['dense_score']:.4f} | "
            f"Sparse: {item['sparse_score']:.4f} | "
            f"Sources: {item['sources']}"
        )

        print(f"Location : {payload.get('location_name')}")
        print(f"File     : {payload.get('source_file')}")
        print(f"Section  : {payload.get('section_number')} - {payload.get('section_title')}")
        print("Content:")
        print((payload.get("content") or "")[:1200])
        print("-" * 80)

    return final_results

In [18]:
hybrid_results = hybrid_search_text_chunks_notebook(
    query_text="hòn khô được biến đến với những điểm nổi bật nào?",
    top_k=3,
    candidate_k=10,
    alpha=0.5,
)

HYBRID TEXT SEARCH: hòn khô được biến đến với những điểm nổi bật nào?
Formula: final_score = 0.5 * dense_score + 0.50 * sparse_score
Top 1 | Final: 5.3181 | Dense: 0.5222 | Sparse: 10.1139 | Sources: ['dense', 'sparse']
Location : None
File     : overview.docx
Section  : 6 - Điểm nổi bật
Content:
Điểm nổi bật thứ nhất là rạn san hô gần bờ. Một số nguồn du lịch mô tả san hô tại Hòn Khô nằm ở vùng nước không quá sâu, phù hợp với hoạt động lặn ống thở khi biển êm và có hướng dẫn an toàn. Đây là lý do Hòn Khô thường được xếp vào nhóm điểm du lịch dành cho khách thích biển xanh, tắm biển và khám phá hệ sinh thái dưới nước.
Điểm nổi bật thứ hai là mùa rong mơ. Từ khoảng tháng 5 đến tháng 7, khu vực Hòn Khô và Nhơn Hải có những thảm rong mơ dưới nước, tạo hiệu ứng như cánh đồng vàng dưới biển. Đây là chất liệu hình ảnh rất phù hợp cho du lịch chụp ảnh, flycam, nội dung truyền thông và các câu trả lời về “mùa nào Hòn Khô đẹp nhất”.
Điểm nổi bật thứ ba là cầu gỗ hoặc đường ven đá nhìn ra biển. 

## 10. Hybrid RAG đơn giản: hỏi text → lấy chunks + ảnh liên quan

Hàm này tìm đồng thời:

```txt
1. Text chunks liên quan từ text_collection
2. Ảnh liên quan từ image_collection bằng hybrid search
```

Dùng để test hướng chatbot sau này.


In [38]:
# =========================
# Query tổng hợp: chunks + images
# =========================

def search_tourism_context(
    query_text: str,
    top_k_text: int = 5,
    top_k_images: int = 5,
    image_alpha: float = 0.5,
    show_image: bool = True,
):
    print("\n" + "#" * 100)
    print("1. TEXT CHUNKS LIÊN QUAN")
    print("#" * 100)

    text_results = search_text_chunks_by_query(
        query_text=query_text,
        top_k=top_k_text,
    )

    print("\n" + "#" * 100)
    print("2. ẢNH LIÊN QUAN")
    print("#" * 100)

    image_results = hybrid_search_images_by_text(
        query_text=query_text,
        top_k=top_k_images,
        candidate_k=max(20, top_k_images * 4),
        alpha=image_alpha,
        show_image=show_image,
    )

    return {
        "text_results": text_results,
        "image_results": image_results,
    }


# Ví dụ:
# context = search_tourism_context(
#     query_text="Hòn Khô có gì đẹp và có nên đi vào mùa hè không?",
#     top_k_text=5,
#     top_k_images=5,
#     image_alpha=0.5,
# )


In [39]:
# Ví dụ:
context = search_tourism_context(
    query_text="Hòn Khô có gì đẹp và có nên đi vào mùa hè không?",
    top_k_text=5,
    top_k_images=5,
    image_alpha=0.5,
)


####################################################################################################
1. TEXT CHUNKS LIÊN QUAN
####################################################################################################
Search text chunks: Hòn Khô có gì đẹp và có nên đi vào mùa hè không?
Top 1 | Score: 0.6810
Location      : None | None
File          : overview.docx
Document type : overview
Section       : 6 - Điểm nổi bật
Chunk         : 1 / 1
Content:
Điểm nổi bật thứ nhất là rạn san hô gần bờ. Một số nguồn du lịch mô tả san hô tại Hòn Khô nằm ở vùng nước không quá sâu, phù hợp với hoạt động lặn ống thở khi biển êm và có hướng dẫn an toàn. Đây là lý do Hòn Khô thường được xếp vào nhóm điểm du lịch dành cho khách thích biển xanh, tắm biển và khám phá hệ sinh thái dưới nước.
Điểm nổi bật thứ hai là mùa rong mơ. Từ khoảng tháng 5 đến tháng 7, khu vực Hòn Khô và Nhơn Hải có những thảm rong mơ dưới nước, tạo hiệu ứng như cánh đồng vàng dưới biển. Đây là chất liệu hình ảnh rất phù 

Top 2 | Final: 0.3195 | Multimodal: 0.0000 | Caption: 0.6389 | Sources: ['caption']
Image ID   : LOC_012_00036_c51557396bc0aed2
File       : 00036_c51557396bc0aed2.jpg
Location   : Hòn Khô | None
Caption VI : Đây là một hòn đảo nằm giữa đại dương: đại dương xanh và xanh tươi; nước rất bình tịnh; có những lượn sóng nhỏ trong nước; trên đỉnh nó có cỏ xanh tươi.
S3 path    : vietnam-tourism/HON_KHO/image/00036_c51557396bc0aed2.jpg
--------------------------------------------------------------------------------


Top 3 | Final: 0.3167 | Multimodal: 0.0000 | Caption: 0.6335 | Sources: ['caption']
Image ID   : LOC_012_00055_97d86c2f6d08b2a9
File       : 00055_97d86c2f6d08b2a9.png
Location   : Hòn Khô | None
Caption VI : Có rất nhiều cây xung quanh Kayak.
S3 path    : vietnam-tourism/HON_KHO/image/00055_97d86c2f6d08b2a9.png
--------------------------------------------------------------------------------


Top 4 | Final: 0.3165 | Multimodal: 0.0000 | Caption: 0.6330 | Sources: ['caption']
Image ID   : LOC_012_00051_b090938a615f6d57
File       : 00051_b090938a615f6d57.jpg
Location   : Hòn Khô | None
Caption VI : Có rất nhiều người đang ở dưới nước, họ đều có ván trượt trên lưng, tất cả đều mặc quần áo.
S3 path    : vietnam-tourism/HON_KHO/image/00051_b090938a615f6d57.jpg
--------------------------------------------------------------------------------


Top 5 | Final: 0.3159 | Multimodal: 0.0000 | Caption: 0.6318 | Sources: ['caption']
Image ID   : LOC_012_00045_e52218f69e569104
File       : 00045_e52218f69e569104.jpg
Location   : Hòn Khô | None
Caption VI : Có một bãi cát, có nhiều người đi bộ trên bãi biển, bên phải bờ biển là một hồ nước lớn, bên bờ biển có nhiều thuyền.
S3 path    : vietnam-tourism/HON_KHO/image/00045_e52218f69e569104.jpg
--------------------------------------------------------------------------------


In [1]:
from qdrant_client import QdrantClient, models


client = QdrantClient(
    url="http://localhost:6333",
)

def delete_by_location_id(collection_name: str, location_id: str):
    location_filter = models.Filter(
        must=[
            models.FieldCondition(
                key="location_id",
                match=models.MatchValue(value=location_id)
            )
        ]
    )

    # Đếm trước khi xóa
    before = client.count(
        collection_name=collection_name,
        count_filter=location_filter,
        exact=True
    ).count

    # Xóa các point có payload location_id tương ứng
    result = client.delete(
        collection_name=collection_name,
        points_selector=models.FilterSelector(
            filter=location_filter
        ),
        wait=True
    )

    # Đếm lại sau khi xóa
    after = client.count(
        collection_name=collection_name,
        count_filter=location_filter,
        exact=True
    ).count

    print(f"Collection: {collection_name}")
    print(f"Trước khi xóa: {before}")
    print(f"Sau khi xóa: {after}")
    print(f"Đã xóa khoảng: {before - after}")

    return result

In [3]:
delete_by_location_id(
    collection_name="text_collection",
    location_id="LOC_077"
)

Collection: text_collection
Trước khi xóa: 39
Sau khi xóa: 0
Đã xóa khoảng: 39


UpdateResult(operation_id=1359, status=<UpdateStatus.COMPLETED: 'completed'>)